# Preprocessing and Binning Configuration Builder (No Train/Test Split)

This notebook loads the pre-encoded feature matrices from the EDA step, applies deterministic feature reductions and transformations (no scaling and no CV here), builds 8 binning configurations for `relationship`, `occupation`, and `workclass`, and saves the resulting full datasets (`X_<config>.csv` and `y.csv`) into the working directory for downstream modeling.

**Leakage control:** cross-validation, scaling, log transforms, and any holdout test split should be performed in the modeling notebook within fold-specific pipelines.


In [1]:
import pandas as pd
import os
import pandas as pd

OUTPUT_DIR = 'Binning_Configs_Full'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Define the local paths for saving the dataframes
DF_PART1_PATH = 'df_encoded_part1.csv'
DF_PART2_PATH = 'df_encoded_part2.csv'

# Define the GitHub URLs
GITHUB_PART1_URL = 'https://raw.githubusercontent.com/BickNutler/Data-Science-Capstone-Two/main/df_encoded_part1.csv'
GITHUB_PART2_URL = 'https://raw.githubusercontent.com/BickNutler/Data-Science-Capstone-Two/main/df_encoded_part2.csv'

# Load data directly from GitHub URLs
df_part1 = pd.read_csv(GITHUB_PART1_URL)
df_part2 = pd.read_csv(GITHUB_PART2_URL)

# Save the dataframes to the predefined local paths
df_part1.to_csv(DF_PART1_PATH, index=False)
df_part2.to_csv(DF_PART2_PATH, index=False)

print(f"Downloaded and saved '{DF_PART1_PATH}' and '{DF_PART2_PATH}'")

Downloaded and saved 'df_encoded_part1.csv' and 'df_encoded_part2.csv'


In [2]:
# 1) Load encoded feature matrices from EDA step
df1 = pd.read_csv(DF_PART1_PATH)
df2 = pd.read_csv(DF_PART2_PATH)
df = pd.concat([df1, df2], ignore_index=True)
print('Loaded df_encoded:', df.shape)


Loaded df_encoded: (48842, 122)


In [3]:
# 2) Drop columns not used for modeling
edu_cols = [c for c in df.columns if c.startswith('education_')]
drop_cols = [
    'fnlwgt',
    'has_capital_gain',
    'income_<=50K',
    'work_schedule_Full-time',
    'work_schedule_Overtime',
    'work_schedule_Part-time',
] + edu_cols
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
print('After drops:', df.shape)


After drops: (48842, 100)


In [4]:
# 3) Deterministic bin/transform operations
# 3a) Native country -> is_US (then drop all native-country columns)
if 'native_country_grouped_United-States' in df.columns:
    df['is_US'] = df['native_country_grouped_United-States'].astype(int)
    df = df.drop(columns=[c for c in df.columns if c.startswith('native_country_grouped_')], errors='ignore')
elif 'native-country_United-States' in df.columns:
    df['is_US'] = df['native-country_United-States'].astype(int)

df = df.drop(columns=[c for c in df.columns if c.startswith('native-country_')], errors='ignore')

# 3b) Race -> is_white (drop all race one-hots)
if 'race_White' in df.columns:
    df['is_white'] = df['race_White'].astype(int)
df = df.drop(columns=[c for c in df.columns if c.startswith('race_')], errors='ignore')

# 3c) Sex -> keep sex_Male as binary (drop sex_Female)
if 'sex_Male' in df.columns:
    df['sex_Male'] = df['sex_Male'].astype(int)
df = df.drop(columns=['sex_Female'], errors='ignore')

# 3d) Marital status -> 3 derived binary flags
married_set = ['marital-status_Married-civ-spouse', 'marital-status_Married-AF-spouse']
never_set = ['marital-status_Never-married']
prev_set = ['marital-status_Divorced', 'marital-status_Separated', 'marital-status_Widowed', 'marital-status_Married-spouse-absent']

df['is_Married'] = df[[c for c in married_set if c in df.columns]].sum(axis=1).clip(0,1).astype(int) if any(c in df.columns for c in married_set) else 0
df['is_Never_married'] = df[[c for c in never_set if c in df.columns]].sum(axis=1).clip(0,1).astype(int) if any(c in df.columns for c in never_set) else 0
df['is_Previously_married'] = df[[c for c in prev_set if c in df.columns]].sum(axis=1).clip(0,1).astype(int) if any(c in df.columns for c in prev_set) else 0
df = df.drop(columns=[c for c in df.columns if c.startswith('marital-status_')], errors='ignore')

# 3e) Boolean -> 0/1
for c in df.columns:
    if df[c].dtype == bool:
        df[c] = df[c].astype(int)

# Drop any pre-existing *_binned_* helper columns (we build binned versions explicitly below)
df = df.drop(columns=[c for c in df.columns if '_binned_' in c], errors='ignore')

print('After deterministic transforms:', df.shape)


After deterministic transforms: (48842, 42)


In [5]:
# 4) Separate features and target
y = df['income_>50K'].astype(int)
X = df.drop(columns=['income_>50K'], errors='ignore')
print('X shape:', X.shape, 'y shape:', y.shape)


X shape: (48842, 41) y shape: (48842,)


In [6]:
# 5) Define bin mappings for relationship, occupation, and workclass
occ_binned_groups = {
    'occupation_High-skill': ['occupation_Exec-managerial', 'occupation_Prof-specialty', 'occupation_Tech-support'],
    'occupation_Mid-skill': ['occupation_Adm-clerical', 'occupation_Sales'],
    'occupation_Blue-collar': ['occupation_Craft-repair', 'occupation_Machine-op-inspct', 'occupation_Transport-moving', 'occupation_Handlers-cleaners'],
    'occupation_Service': ['occupation_Other-service', 'occupation_Priv-house-serv'],
    'occupation_Protective': ['occupation_Protective-serv', 'occupation_Armed-Forces'],
}

rel_binned_groups = {
    'relationship_Spouse-present': ['relationship_Husband', 'relationship_Wife']
}

wc_binned_groups = {
    'workclass_Government': ['workclass_Federal-gov', 'workclass_Local-gov', 'workclass_State-gov'],
    'workclass_Self-employed': ['workclass_Self-emp-inc', 'workclass_Self-emp-not-inc'],
    'workclass_no_work_income': ['workclass_Never-worked', 'workclass_Without-pay'],
}

removed_occ = [
  'occupation_Adm-clerical','occupation_Armed-Forces','occupation_Craft-repair','occupation_Exec-managerial',
  'occupation_Handlers-cleaners','occupation_Machine-op-inspct','occupation_Other-service','occupation_Priv-house-serv',
  'occupation_Prof-specialty','occupation_Protective-serv','occupation_Sales','occupation_Tech-support','occupation_Transport-moving'
]
removed_rel = ['relationship_Husband','relationship_Wife']
removed_wc = ['workclass_Federal-gov','workclass_Local-gov','workclass_Never-worked','workclass_Self-emp-inc','workclass_Self-emp-not-inc','workclass_State-gov','workclass_Without-pay']


In [7]:
# 5b) Apply bin mappings to create 8 binning configurations
def apply_binning(X_in, bin_occ=False, bin_rel=False, bin_wc=False):
    Xc = X_in.copy()
    if bin_occ:
        for new_col, old_cols in occ_binned_groups.items():
            Xc[new_col] = Xc[old_cols].sum(axis=1).clip(0,1).astype(int)
        Xc = Xc.drop(columns=removed_occ, errors='ignore')
    if bin_rel:
        for new_col, old_cols in rel_binned_groups.items():
            Xc[new_col] = Xc[old_cols].sum(axis=1).clip(0,1).astype(int)
        Xc = Xc.drop(columns=removed_rel, errors='ignore')
    if bin_wc:
        for new_col, old_cols in wc_binned_groups.items():
            Xc[new_col] = Xc[old_cols].sum(axis=1).clip(0,1).astype(int)
        Xc = Xc.drop(columns=removed_wc, errors='ignore')
    return Xc

configs_flags = {
    'no_binning': (False, False, False),
    'occupation_binned': (True, False, False),
    'relationship_binned': (False, True, False),
    'workclass_binned': (False, False, True),
    'occupation_relationship_binned': (True, True, False),
    'occupation_workclass_binned': (True, False, True),
    'relationship_workclass_binned': (False, True, True),
    'all_binned': (True, True, True),
}

X_configs = {name: apply_binning(X, *flags) for name, flags in configs_flags.items()}
for name, Xcfg in X_configs.items():
    print(name, Xcfg.shape)


no_binning (48842, 41)
occupation_binned (48842, 33)
relationship_binned (48842, 40)
workclass_binned (48842, 37)
occupation_relationship_binned (48842, 32)
occupation_workclass_binned (48842, 29)
relationship_workclass_binned (48842, 36)
all_binned (48842, 28)


In [8]:
# 6) Save the 8 unique binning combos of X, and save y
y.to_csv(os.path.join(OUTPUT_DIR, 'y.csv'), index=False, header=['income_>50K'])
for name, Xcfg in X_configs.items():
    out_path = os.path.join(OUTPUT_DIR, f'X_{name}.csv')
    Xcfg.to_csv(out_path, index=False)
    print('Saved', out_path)


Saved Binning_Configs_Full/X_no_binning.csv
Saved Binning_Configs_Full/X_occupation_binned.csv
Saved Binning_Configs_Full/X_relationship_binned.csv
Saved Binning_Configs_Full/X_workclass_binned.csv
Saved Binning_Configs_Full/X_occupation_relationship_binned.csv
Saved Binning_Configs_Full/X_occupation_workclass_binned.csv
Saved Binning_Configs_Full/X_relationship_workclass_binned.csv
Saved Binning_Configs_Full/X_all_binned.csv
